# Predicting Premier League Home Wins
## 1. Setup

In [3]:
from pathlib import Path
import urllib.request
import pandas as pd

DATA_DIR = Path("data") #stores in data
SEASONS = ["1516", "1617", "1718", "1819", "1920",
           "2021", "2122", "2223", "2324", "2425"] #which seasons ill be working with

## 2. Data loading

In [4]:
DATA_DIR.mkdir(exist_ok=True)
#download data and store as csv file
for season in SEASONS:
    file_path = DATA_DIR / f"E0_{season}.csv"
    if file_path.exists():
        print("Already have", file_path)
    else:
        url = f"https://football-data.co.uk/mmz4281/{season}/E0.csv"
        urllib.request.urlretrieve(url, file_path)
        print("Downloaded", file_path)

Already have data\E0_1516.csv
Already have data\E0_1617.csv
Already have data\E0_1718.csv
Already have data\E0_1819.csv
Already have data\E0_1920.csv
Already have data\E0_2021.csv
Already have data\E0_2122.csv
Already have data\E0_2223.csv
Already have data\E0_2324.csv
Already have data\E0_2425.csv


In [5]:
frames = []
for season in SEASONS:
    df_season = pd.read_csv(DATA_DIR / f"E0_{season}.csv") #read csv files
    df_season["Season"] = season
    frames.append(df_season)

matches = pd.concat(frames, ignore_index=True) #combine
matches.shape 

C:\Users\adam1_4sss501\AppData\Local\Temp\ipykernel_6276\3318796406.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_season["Season"] = season


(3800, 154)

## 3. Data audit

In [6]:
missing_pct = matches.isna().mean() * 100 #calculate missing values
missing_pct.sort_values(ascending=False).head(20)

1XBD       90.236842
1XBH       90.236842
1XBA       90.236842
BFE>2.5    90.078947
BFE<2.5    90.078947
BFA        90.026316
BFH        90.026316
BFD        90.026316
1XBCD      90.000000
1XBCH      90.000000
BFECH      90.000000
1XBCA      90.000000
BFEAHA     90.000000
BFCH       90.000000
BFCD       90.000000
BFCA       90.000000
BFECAHA    90.000000
BFECAHH    90.000000
BFEH       90.000000
BFEAHH     90.000000
dtype: float64

In [7]:
key_cols = ["Date", "HomeTeam", "AwayTeam", "FTR",
            "FTHG", "FTAG", "HS", "AS", "HST", "AST", "HC", "AC",
            "B365H", "B365D", "B365A",
            "AvgH", "AvgD", "AvgA", "BbAvH", "BbAvD", "BbAvA"]
missing_pct[key_cols] #show missing data percent for my columns

Date         0.0
HomeTeam     0.0
AwayTeam     0.0
FTR          0.0
FTHG         0.0
FTAG         0.0
HS           0.0
AS           0.0
HST          0.0
AST          0.0
HC           0.0
AC           0.0
B365H        0.0
B365D        0.0
B365A        0.0
AvgH        40.0
AvgD        40.0
AvgA        40.0
BbAvH       60.0
BbAvD       60.0
BbAvA       60.0
dtype: float64

The average odds column names changed between seasons (BbAvH to AvgH) so Bet 365 odds (B365H/D/A) were used instead as they are complete for all 10 seasons

In [8]:
keep_cols = ["Season", "Date", "HomeTeam", "AwayTeam", "FTR",
             "FTHG", "FTAG", "HS", "AS", "HST", "AST", "HC", "AC",
             "B365H", "B365D", "B365A"] #removed useless columns
df = matches[keep_cols].copy()
df.head() #first five rows

,Season,Date,HomeTeam,AwayTeam,FTR,FTHG,FTAG,HS,AS,HST,AST,HC,AC,B365H,B365D,B365A
0,1516,08/08/2015,Bournemouth,Aston Villa,A,0,1,11,7,2,3,6,3,2.00,3.6,4.00
1,1516,08/08/2015,Chelsea,Swansea,D,2,2,11,18,3,10,4,8,1.36,5.0,11.00
2,1516,08/08/2015,Everton,Watford,D,2,2,10,11,5,5,8,2,1.70,3.9,5.50
3,1516,08/08/2015,Leicester,Sunderland,H,4,2,19,10,8,5,6,3,1.95,3.5,4.33
4,1516,08/08/2015,Man United,Tottenham,H,1,0,9,9,1,4,1,2,1.65,4.0,6.00


In [9]:
df.dtypes #data types

Season          str
Date            str
HomeTeam        str
AwayTeam        str
FTR             str
FTHG          int64
FTAG          int64
HS            int64
AS            int64
HST           int64
AST           int64
HC            int64
AC            int64
B365H       float64
B365D       float64
B365A       float64
dtype: object

In [10]:
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, format="mixed") #text to real dates and time format fix
df = df.sort_values("Date").reset_index(drop=True) #order match 
df.groupby("Season")["Date"].agg(["min", "max"]) #first vs last

,min,max
Season,,
1516,2015-08-08,2016-05-17
1617,2016-08-13,2017-05-21
1718,2017-08-11,2018-05-13
1819,2018-08-10,2019-05-12
1920,2019-08-09,2020-07-26
2021,2020-09-12,2021-05-23
2122,2021-08-13,2022-05-22
2223,2022-08-05,2023-05-28
2324,2023-08-11,2024-05-19


In [11]:
df.duplicated(subset=["Date", "HomeTeam", "AwayTeam"]).sum() #duplicates

np.int64(0)

In [12]:
#target is 1 is home win and 0 is draw or away win
df["HomeWin"] = (df["FTR"] == "H").astype(int)

print(df["FTR"].value_counts(normalize=True))
print(df["HomeWin"].value_counts(normalize=True))

FTR
H    0.445000
A    0.321842
D    0.233158
Name: proportion, dtype: float64
HomeWin
0    0.555
1    0.445
Name: proportion, dtype: float64


No duplicated were found. Home teams won 44.5% of all the matches so the target is pretty balanced (55.5/44.5). If we always predicted "not home win", this would be 55.5% accurate.

In [13]:
df.describe().round(2) #simple quick summary of number columns

,Date,FTHG,FTAG,HS,AS,HST,AST,HC,AC,B365H,B365D,B365A,HomeWin
count,3800,3800.00,3800.00,3800.00,3800.00,3800.00,3800.00,3800.00,3800.00,3800.00,3800.00,3800.00,3800.00
mean,2020-07-08 15:48:07.578947,1.55,1.28,13.92,11.50,4.78,4.02,5.72,4.72,2.96,4.28,4.76,0.44
min,2015-08-08 00:00:00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.06,2.80,1.12,0.00
25%,2017-12-25 06:00:00,1.00,0.00,10.00,8.00,3.00,2.00,3.00,3.00,1.65,3.48,2.20,0.00
50%,2020-08-19 00:00:00,1.00,1.00,13.00,11.00,5.00,4.00,5.00,4.00,2.25,3.75,3.30,0.00
75%,2023-01-21 00:00:00,2.00,2.00,17.00,15.00,6.00,5.00,8.00,6.00,3.25,4.50,5.50,1.00
max,2025-05-25 00:00:00,9.00,9.00,37.00,37.00,17.00,15.00,19.00,19.00,23.00,17.00,41.00,1.00
std,NaN,1.32,1.21,5.74,5.05,2.65,2.37,3.11,2.78,2.25,1.43,4.26,0.50


In [14]:
df[(df["HS"] == 0) | (df["AS"] == 0)] #checking 0 shots incase of data error

,Season,Date,HomeTeam,AwayTeam,FTR,FTHG,FTAG,HS,AS,HST,AST,HC,AC,B365H,B365D,B365A,HomeWin
1054,1718,2018-03-10,Huddersfield,Swansea,D,0,0,30,0,4,0,12,0,2.54,3.1,3.20,0
1422,1819,2019-03-02,Bournemouth,Man City,A,0,1,0,23,0,7,0,14,13.00,7.5,1.22,0


There were no impossible values were found such as negative goals and all odds above 1.0. Extreme values like 9 goals or zero shot performances and odds like 41.0 are actual real one sided games and performances. 

## 4. Preprocessing and feature engineering 

In [ ]:
# One row per team per match. home teams view and away teams view
home = pd.DataFrame({
    "MatchID": df.index, #links back to og match
    "Date": df["Date"],
    "Team": df["HomeTeam"],
    "GoalsFor": df["FTHG"],
    "GoalsAgainst": df["FTAG"],
    "Shots": df["HS"],
    "ShotsOnTarget": df["HST"],
    "Corners": df["HC"],
    "Points": df["FTR"].map({"H": 3, "D": 1, "A": 0}),
})

away = pd.DataFrame({
    "MatchID": df.index,
    "Date": df["Date"],
    "Team": df["AwayTeam"],
    "GoalsFor": df["FTAG"],
    "GoalsAgainst": df["FTHG"],
    "Shots": df["AS"],
    "ShotsOnTarget": df["AST"],
    "Corners": df["AC"],
    "Points": df["FTR"].map({"H": 0, "D": 1, "A": 3}),
})

team_games = pd.concat([home, away]).sort_values(["Team", "Date"]).reset_index(drop=True) #put all together
team_games.head(10) #show only first few

,MatchID,Date,Team,GoalsFor,GoalsAgainst,Shots,ShotsOnTarget,Corners,Points
0,6,2015-08-09,Arsenal,0,2,22,6,5,0
1,17,2015-08-16,Arsenal,2,1,20,7,6,3
2,29,2015-08-24,Arsenal,0,0,19,5,7,1
3,35,2015-08-29,Arsenal,1,0,22,9,9,3
4,43,2015-09-12,Arsenal,2,0,29,12,13,3
5,55,2015-09-19,Arsenal,0,2,9,2,5,0
6,60,2015-09-26,Arsenal,5,2,26,12,5,3
7,77,2015-10-04,Arsenal,3,0,12,5,3,3
8,80,2015-10-17,Arsenal,3,0,17,6,9,3
9,90,2015-10-24,Arsenal,2,1,20,5,7,3


In [22]:
stat_cols = ["GoalsFor", "GoalsAgainst", "Shots", "ShotsOnTarget", "Corners", "Points"]


form = (
    team_games.groupby("Team")[stat_cols]
    .transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean()) 
)
#for each team average last 5 games not including current match preventing data leakage.
#Early games use however many past games exist

form.columns = ["Form_" + c for c in stat_cols] #new columns

team_games = pd.concat([team_games, form], axis=1)
team_games.head(7)

,MatchID,Date,Team,GoalsFor,GoalsAgainst,Shots,ShotsOnTarget,Corners,Points,Form_GoalsFor,...,Form_Shots,Form_ShotsOnTarget,Form_Corners,Form_Points,Form_GoalsFor,Form_GoalsAgainst,Form_Shots,Form_ShotsOnTarget,Form_Corners,Form_Points
0,6,2015-08-09,Arsenal,0,2,22,6,5,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,17,2015-08-16,Arsenal,2,1,20,7,6,3,0.000000,...,22.000000,6.00,5.00,0.000000,0.000000,2.00,22.000000,6.00,5.00,0.000000
2,29,2015-08-24,Arsenal,0,0,19,5,7,1,1.000000,...,21.000000,6.50,5.50,1.500000,1.000000,1.50,21.000000,6.50,5.50,1.500000
3,35,2015-08-29,Arsenal,1,0,22,9,9,3,0.666667,...,20.333333,6.00,6.00,1.333333,0.666667,1.00,20.333333,6.00,6.00,1.333333
4,43,2015-09-12,Arsenal,2,0,29,12,13,3,0.750000,...,20.750000,6.75,6.75,1.750000,0.750000,0.75,20.750000,6.75,6.75,1.750000
5,55,2015-09-19,Arsenal,0,2,9,2,5,0,1.000000,...,22.400000,7.80,8.00,2.000000,1.000000,0.60,22.400000,7.80,8.00,2.000000
6,60,2015-09-26,Arsenal,5,2,26,12,5,3,1.000000,...,19.800000,7.00,8.00,2.000000,1.000000,0.60,19.800000,7.00,8.00,2.000000
